# Análise exploratória tabela estabelecimentos RAIS

## Setups

### Definição de constantes

In [ ]:
GCP_PROJECT_ID = "pure-league-482018-a7"
DATASET_NAME = "basedosdados"
SCHEMA_NAME = "br_me_rais"
TABLE= "microdados_estabelecimentos"

### Configurações de ambiente

In [2]:
from google.cloud import bigquery
from pathlib import Path
import os
import warnings

actual_path = Path().absolute()
os.chdir(actual_path.parent.parent.parent)

warnings.filterwarnings("ignore", category=UserWarning, module="google.cloud.bigquery")
client = bigquery.Client(project=GCP_PROJECT_ID)

## análise de metadados

In [ ]:
query_metadata_t1 = f"""
    SELECT *
    FROM `{DATASET_NAME}.{SCHEMA_NAME}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{TABLE}';
"""

df_metadata_t1 = client.query(query_metadata_t1).to_dataframe()
df_metadata_t1

,table_catalog,table_schema,table_name,column_name,ordinal_position,is_nullable,data_type,is_generated,generation_expression,is_stored,...,is_system_defined,is_partitioning_column,clustering_ordinal_position,collation_name,column_default,rounding_mode,data_policies,data_governance_tags,policy_tags,async_generation_status
0,basedosdados,br_me_rais,microdados_estabelecimentos,ano,1,YES,INT64,NEVER,NaN,NaN,...,NO,YES,<NA>,NULL,NULL,NaN,[],[],[],None
1,basedosdados,br_me_rais,microdados_estabelecimentos,sigla_uf,2,YES,STRING,NEVER,NaN,NaN,...,NO,NO,1,NULL,NULL,NaN,[],[],[],None
2,basedosdados,br_me_rais,microdados_estabelecimentos,id_municipio,3,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
3,basedosdados,br_me_rais,microdados_estabelecimentos,quantidade_vinculos_ativos,4,YES,INT64,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
4,basedosdados,br_me_rais,microdados_estabelecimentos,quantidade_vinculos_clt,5,YES,INT64,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
5,basedosdados,br_me_rais,microdados_estabelecimentos,quantidade_vinculos_estatutarios,6,YES,INT64,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
6,basedosdados,br_me_rais,microdados_estabelecimentos,natureza_estabelecimento,7,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
7,basedosdados,br_me_rais,microdados_estabelecimentos,natureza_juridica,8,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
8,basedosdados,br_me_rais,microdados_estabelecimentos,tamanho_estabelecimento,9,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
9,basedosdados,br_me_rais,microdados_estabelecimentos,tipo_estabelecimento,10,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None


## Análise de preenchimento

In [ ]:
count_list = [f"COUNTIF({col} IS NOT NULL) / COUNT(*) AS {col}_filled_pct" for col in df_metadata_t1['column_name'].values]
count_list_str = ",\n    ".join(count_list)

query_missingness = f"""
SELECT
    {count_list_str}
FROM 
`basedosdados.br_me_rais.{TABLE}`
"""

df2 = client.query(query_missingness).to_dataframe()
df2_long = df2.T.reset_index()
df2_long.columns = ["field", "value"]
df2_long.sort_values(by="value", ascending=False)

,field,value
0,ano_filled_pct,1.000000
1,sigla_uf_filled_pct,1.000000
3,quantidade_vinculos_ativos_filled_pct,1.000000
9,tipo_estabelecimento_filled_pct,1.000000
8,tamanho_estabelecimento_filled_pct,1.000000
2,id_municipio_filled_pct,0.995170
15,cnae_1_filled_pct,0.950514
13,indicador_rais_negativa_filled_pct,0.943949
7,natureza_juridica_filled_pct,0.905489
11,indicador_pat_filled_pct,0.867781


#### Registros a partir de 2020

In [ ]:
count_list = [f"COUNTIF({col} IS NOT NULL) / COUNT(*) AS {col}_filled_pct" for col in df_metadata_t1['column_name'].values]
count_list_str = ",\n    ".join(count_list)

query_missingness = f"""
SELECT
    {count_list_str}
FROM 
`basedosdados.br_me_rais.{TABLE}`
WHERE ano >= 2020
"""

df2 = client.query(query_missingness).to_dataframe()
df2_long = df2.T.reset_index()
df2_long.columns = ["field", "value"]
df2_long.sort_values(by="value", ascending=False)

,field,value
0,ano_filled_pct,1.000000
1,sigla_uf_filled_pct,1.000000
3,quantidade_vinculos_ativos_filled_pct,1.000000
4,quantidade_vinculos_clt_filled_pct,1.000000
5,quantidade_vinculos_estatutarios_filled_pct,1.000000
9,tipo_estabelecimento_filled_pct,1.000000
8,tamanho_estabelecimento_filled_pct,1.000000
12,indicador_simples_filled_pct,1.000000
11,indicador_pat_filled_pct,1.000000
10,indicador_cei_vinculado_filled_pct,1.000000


## análise de distribuição de valores

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import webbrowser


def get_value_counts(client, table, col):
    query = f"""
    SELECT
        `{col}` AS value,
        COUNT(*) AS freq
    FROM `{table}`
    GROUP BY `{col}`
    ORDER BY freq DESC
    
    """
    df = client.query(query).to_dataframe()
    df["value"] = df["value"].astype(str)  # garante eixo categórico consistente
    return df

columns = list(df_metadata_t1['column_name'].values)
table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"

# Coleta os value_counts de cada coluna (uma query por coluna)
value_counts_dict = {}
for col in columns:
    print(f"Consultando: {col}")
    value_counts_dict[col] = get_value_counts(client, table, col)

# Monta o grid vertical: 1 gráfico por linha
n = len(columns)
fig = make_subplots(
    rows=n, cols=1,
    subplot_titles=[f"Distribuição: {col}" for col in columns],
    vertical_spacing=0.4 / n  

for i, col in enumerate(columns, start=1):
    df_vc = value_counts_dict[col]
    fig.add_trace(
        go.Bar(
            x=df_vc["value"],
            y=df_vc["freq"],
            name=col,
            showlegend=False
        ),
        row=i, col=1
    )
    fig.update_xaxes(tickangle=45, row=i, col=1)

# Altura por gráfico ~350px, permitindo scroll vertical no notebook/HTML
fig.update_layout(
    height=350 * n,
    width=1000,
    title_text=f"Distribuição de frequência por variável da tabela '{TABLE}'",
    showlegend=False
)

# Salva como HTML
output_path = os.path.abspath(f"data/outputs/distribuicao_{TABLE}.html")
fig.write_html(output_path, include_plotlyjs="cdn")
webbrowser.open(f"file://{output_path}")

Consultando: ano
Consultando: sigla_uf
Consultando: id_municipio
Consultando: quantidade_vinculos_ativos
Consultando: quantidade_vinculos_clt
Consultando: quantidade_vinculos_estatutarios
Consultando: natureza_estabelecimento
Consultando: natureza_juridica
Consultando: tamanho_estabelecimento
Consultando: tipo_estabelecimento
Consultando: indicador_cei_vinculado
Consultando: indicador_pat
Consultando: indicador_simples
Consultando: indicador_rais_negativa
Consultando: indicador_atividade_ano
Consultando: cnae_1
Consultando: cnae_2
Consultando: cnae_2_subclasse
Consultando: subsetor_ibge
Consultando: subatividade_ibge
Consultando: cep
Consultando: bairros_sp
Consultando: distritos_sp
Consultando: bairros_fortaleza
Consultando: bairros_rj
Consultando: regioes_administrativas_df


True

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import webbrowser


def get_value_counts(client, table, col):
    query = f"""
    SELECT
        `{col}` AS value,
        COUNT(*) AS freq
    FROM `{table}`
    GROUP BY `{col}`
    ORDER BY freq DESC
    
    """
    df = client.query(query).to_dataframe()
    df["value"] = df["value"].astype(str)  # garante eixo categórico consistente
    return df

columns = df_metadata_t1['column_name'].values

table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"

# Coleta os value_counts de cada coluna (uma query por coluna)
value_counts_dict = {}
for col in columns:
    print(f"Consultando: {col}")
    value_counts_dict[col] = get_value_counts(client, table, col)

# Monta o grid vertical: 1 gráfico por linha
n = len(columns)
fig = make_subplots(
    rows=n, cols=1,
    subplot_titles=[f"Distribuição: {col}" for col in columns],
    vertical_spacing=0.4 / n  # espaçamento proporcional para não sobrepor títulos
)

for i, col in enumerate(columns, start=1):
    df_vc = value_counts_dict[col]
    fig.add_trace(
        go.Bar(
            x=df_vc["value"],
            y=df_vc["freq"],
            name=col,
            showlegend=False
        ),
        row=i, col=1
    )
    fig.update_xaxes(tickangle=45, row=i, col=1)

# Altura por gráfico ~350px, permitindo scroll vertical no notebook/HTML
fig.update_layout(
    height=350 * n,
    width=1000,
    title_text=f"Distribuição de frequência por variável da tabela '{TABLE}'",
    showlegend=False
)

# Salva como HTML
output_path = os.path.abspath(f"data/outputs/distribuicao_{TABLE}.html")
fig.write_html(output_path, include_plotlyjs="cdn")
webbrowser.open(f"file://{output_path}")

Consultando: ano
Consultando: sigla_uf
Consultando: id_municipio
Consultando: quantidade_vinculos_ativos
Consultando: quantidade_vinculos_clt
Consultando: quantidade_vinculos_estatutarios
Consultando: natureza_estabelecimento
Consultando: natureza_juridica
Consultando: tamanho_estabelecimento
Consultando: tipo_estabelecimento
Consultando: indicador_cei_vinculado
Consultando: indicador_pat
Consultando: indicador_simples
Consultando: indicador_rais_negativa
Consultando: indicador_atividade_ano
Consultando: cnae_1
Consultando: cnae_2
Consultando: cnae_2_subclasse
Consultando: subsetor_ibge
Consultando: subatividade_ibge
Consultando: cep
Consultando: bairros_sp
Consultando: distritos_sp
Consultando: bairros_fortaleza
Consultando: bairros_rj
Consultando: regioes_administrativas_df


True